# Generate SFT Dataset

## Load Library

In [293]:
import os
import re
import json
import base64
import requests
import time
import mimetypes
import hashlib
from tqdm import tqdm
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

## Configuration

In [294]:
GEN_TEST_WITH_MODEL = False

### Define Path

In [295]:
BASE_DIR = Path(".")

METADATA_PATH = (
    BASE_DIR
    / "metadata"
    / "mkn_house_metadata_sample.json"
)

IMAGE_BASE_DIR = (
    BASE_DIR
    / "data"
    / "mkn_img"
)

OUTPUT_BASE_DIR = (
    BASE_DIR
    / "data"
    / "sft_dataset"
)

OUTPUT_BASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_JSONL = OUTPUT_BASE_DIR / "train.jsonl"
VAL_JSONL = OUTPUT_BASE_DIR / "val.jsonl"
TEST_JSONL = OUTPUT_BASE_DIR / "test.jsonl"
ALL_JSONL = OUTPUT_BASE_DIR / "all.jsonl"

CACHE_PATH = OUTPUT_BASE_DIR / "cache.json"

### OpenRouter

In [296]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openai/gpt-5-mini"

### OpenRouter Config

In [297]:
TEMPERATURE = 0.1
MAX_TOKENS = 10000
TIMEOUT = 120
MAX_RETRIES = 3
RETRY_SLEEP_SEC = 2
MAX_WORKERS = 2

In [298]:
print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_BASE_DIR:", OUTPUT_BASE_DIR)
print("METADATA_PATH:", METADATA_PATH)
print("GEN_TEST_WITH_MODEL:", GEN_TEST_WITH_MODEL)

MODEL_NAME: openai/gpt-5-mini
OUTPUT_BASE_DIR: data\sft_dataset
METADATA_PATH: metadata\mkn_house_metadata_sample.json
GEN_TEST_WITH_MODEL: False


### OpenRouter Prompt

In [299]:
# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT_REASON = """
Kamu adalah ahli material bangunan yang bertugas menjelaskan ciri visual yang mendukung label prediksi material bangunan.

TUGAS:
Amati foto rumah yang diberikan, perhatikan visual dari atap,dinding,dan lantai. Tulis deskripsi visual yang menjelaskan alasan
mengapa komponen diprediksi sebagai label tersebut. Jadi tugas kamu adalah menjustifikasi label  prediksinya.

GUNAKAN CIRI VISUAL SEPERTI :
- tekstur
- pola
- bentuk permukaan
- susunan material
- karakter visual yang terlihat
- letak terlihatnya dimana

CONTOH GAYA BAHASANYA :
- terlihat menggunakan
- tampak tersusun dari
- memiliki pola
- memiliki permukaan
- tampak menggunakan

PENTING:
- Label prediksi SUDAH ditentukan.
- Jangan melakukan klasifikasi ulang.
- Jangan mengubah label.
- jangan menyebut status.
- JANGAN BERHALUSINASi
- JANGAN menybutkan material lain yang tidak relevan dengan label prediksi pada kalimat penjelasan.
- hindari kata kata yang abstrak dan tidak terlihat langsung secara visual
- jangan menjelaskan detail yang tidak relevan dengan kategori label prediksi.
- jangan menjelaskan terkait kualitas rumah, tapi lebih ke ciri visual label prediksi nya.
- Untuk label Tembok: Jika tertutup plester/cat, jelaskan permukaan bidang yang solid, rata, dan keras yang mengindikasikan konstruksi pasangan bata/batako permanen.- buat lebih konsisten penjelasan labelnya, tapi tetap mewakili visualnya.
- untuk komponen atap jangan memperhatikan kanopi kalau atap utamanya masih kelihatan. Kalau atap utamanya tidak kelihatan maka label prediksi pasti materila kanopi. Sesuaikan dengan prediksi
- untuk komponen lantai jika lantai di lapisi karpet jangan mendeskripsikan karpetnya tapi cari terlebih dahulu area tepi yang mengekspos fondasi aslinya. tapi kalau prediksinya Parket/vinil/karpet barulah deskripsikan karpetnya
- Jangan mengeluarkan output prediksi dan status, keluarkan hanya penjelasan per komponen
 
ATURAN:
- Tulis tepat SATU kalimat.
- Panjang penjelasan 12-20 kata.
- Jelaskan HANYA komponen yang sedang diproses. Jangan membahas objek lain seperti ventilasi, jendela, pintu.
- Abaikan objek selain komponen target.
- Anggap label prediksi sudah benar.
- Material boleh disebut jika menjadi bagian dari alasan visual.
- penjelasan visual harus di tekankan kenapa dia masuk ke kategori label itu, jelaskan ciri ciri visualnya dan terlihat dimana
- Jangan gunakan tanda baca koma (,), titik dua(:), garis miring (/) dalam penjelasan, gunakan kata penghubung seperti "dan", "yang memiliki", "dengan", dst.

OUTPUT WAJIB JSON VALID:
OUTPUT HARUS BERUPA JSON valid dengan format berikut:
{{
"atap":"Atap rumah terlihat menggunakan lembaran logam bergelombang pada bidang atap utama.",
"dinding":"Dinding luar rumah tampak memiliki permukaan bidang solid dan rata.",
"lantai":""
}}

Jika Generate alasan = False:
isi dengan string kosong ("").
"""

# ============================================================
# DESKRIPSI SKEMA
# ============================================================

SCHEMA_DESC = {
    "A": "Exterior Only (foto tampak luar)",
    "B": "Interior Only (foto tampak dalam)",
    "C": "Multi-Image (foto tampak luar + tampak dalam)",
}


# ============================================================
# BUILD USER PROMPT
# ============================================================

def build_reason_user_text(
    house_id: str,
    schema: str,
    components: dict,
):

    component_text = ""

    for comp in ["atap", "dinding", "lantai"]:

        c = components[comp]

        component_text += f"""
{comp.capitalize()}
- Label prediksi  : {c["prediksi_label"]}
- Generate alasan : {c["needs_api"]}

"""

    USER_PROMPT = f"""
Berikut data rumah yang harus dijelaskan.

HOUSE
- ID    : {house_id}
- Skema : {SCHEMA_DESC.get(schema, schema)}

KOMPONEN TARGET
{component_text}

TUGAS :
- Tulis penjelasan visual mengapa komponen diprediksi sebagai label tersebut.
- Fokus hanya pada komponen target.
- Abaikan objek lain pada gambar.
- Gunakan observasi visual yang terlihat langsung.

AWALI KALIMAT DENGAN :
- Atap rumah ...
- Dinding luar rumah ...
- Lantai dalam rumah ...

CONTOH :
- Kayu/papan → Lantai dalam rumah terlihat menggunakan papan kayu panjang dengan sambungan antar papan dan tekstur serat kayu.
- Genteng → Atap rumah terlihat memiliki pola genteng bergelombang berwarna merah kecoklatan.
- Anyaman bambu → Dinding luar rumah tampak tersusun dari anyaman bambu dengan pola anyaman yang jelas pada permukaan dinding.
- Seng → Atap rumah terlihat menggunakan lembaran logam memanjang dengan pola bergelombang dan permukaan sedikit reflektif.
- Asbes → Atap rumah terlihat memiliki tepi atap bergelombang tebal, membulat tumpul, dan abu-abu kusam memastikan penggunaan material lembaran asbes.
- Keramik → Lantai dalam rumah terlihat menggunakan ubin mengilap dengan pola teratur dan garis nat yang jelas.
- Semen/bata merah → Lantai abu-abu kusam, kasar, dan tanpa nat menunjukkan karakteristik visual plesteran semen tanpa pelapis.

PENTING:
- - WAJIB KELUARKAN PENJELASAN JIKA PREDIKSI SELAIN "Tidak Terdeteksi"
- Jika Generate alasan = False → isi string kosong ("")
- Kembalikan HANYA JSON valid
- JANGAN MENGELUARKAN OUTPUT APAPUN SELAIN PENJELASAN
- JANGAN MENGELUARKAN LABEL PREDIKSI, STATUS, ATAU INFORMASI LAINNYA

OUTPUT WAJIB JSON VALID:
OUTPUT HARUS BERUPA JSON valid dengan format berikut:
{{
"atap":"Atap rumah terlihat menggunakan lembaran logam bergelombang pada bidang atap utama.",
"dinding":"Dinding luar rumah tampak memiliki permukaan bidang solid dan rata.",
"lantai":""
}}
"""

    return USER_PROMPT.strip()

### SFT Prompt

In [ ]:
SYSTEM_PROMPT_SFT = """Kamu adalah sistem AI inspeksi material bangunan untuk validasi data DTSEN. Tugasmu adalah menganalisis citra rumah secara visual untuk mengidentifikasi material terluasn pada komponen bangunan yaitu Atap, Dinding, dan Lantai, mengklasifikasikan jenis material yang teramati, membandingkannya dengan data referensi DTSEN, serta memberikan penjelasan berbasis bukti visual pada setiap komponen sebagai alasan hasil validasi.

Analisis komponen harus mengikuti instruksi tugas pada pesan user. Hanya lakukan prediksi pada komponen yang diizinkan untuk dianalisis sesuai skema input. Jika suatu komponen dinyatakan tidak dianalisis
pada instruksi user, maka komponen tersebut wajib mengikuti aturan yang ditetapkan user.

exterior = tampak luar
interior = tampak dalam

LABEL MATERIAL RESMI DTSEN :
Gunakan HANYA satu label per komponen dari daftar berikut. Penulisan harus persis sama, termasuk huruf kapital, spasi, dan garis miring.
Dilarang penggabungan label atau penggunaan label di luar daftar.

Komponen Atap:
Beton
Genteng
Seng
Asbes
Bambu
Kayu/sirap
Jerami/ijuk/daun-daunan/rumbia
Lainnya
Tidak terdeteksi

Komponen Dinding:
Tembok
Plesteran anyaman bambu/kawat
Kayu/papan/gypsum/GRC/calciboard
Anyaman bambu
Batang kayu
Bambu
Lainnya
Tidak terdeteksi

Komponen Lantai:
Marmer/granit
Keramik
Parket/vinil/karpet
Ubin/tegel/teraso
Kayu/papan
Semen/bata merah
Bambu
Tanah
Lainnya
Tidak terdeteksi

Jika komponen yang seharusnya dianalisis tidak terlihat, tertutup objek lain, berada di luar area citra, atau bukan visual rumah, gunakan:
Prediksi : Tidak terdeteksi
Status   : Tidak teridentifikasi
Alasan : (sesuaikan dengan komponen)
"Atap": "Variabel atap tidak dapat diidentifikasi karena komponen atap pada foto rumah tampak luar tidak terlihat."
"Dinding": "Variabel dinding tidak dapat diidentifikasi karena komponen dinding pada foto rumah tampak luar tidak terlihat."
"Lantai":"Variabel lantai tidak dapat diidentifikasi karena komponen lantai pada foto rumah tampak dalam tidak terlihat.",

KETENTUAN STATUS :
Pilih tepat satu status untuk setiap komponen yang dianalisis:
Sesuai : prediksi visual identik atau konsisten dengan data referensi DTSEN.
Tidak sesuai : prediksi visual berbeda dengan data referensi DTSEN.
Tidak teridentifikasi : digunakan ketika prediksi adalah "Tidak terdeteksi",yaitu saat komponen tidak terlihat, bukti visual tidak cukup, atau komponen tidak tersedia sesuai konteks citra dan instruksi user.


KETENTUAN ALASAN :
Untuk setiap komponen, hasilkan penjelasan visual untuk field Alasan:
- Komponen terdeteksi (Prediksi ≠ "Tidak terdeteksi"): Tulis minimal 10–20 kata yang menjelaskan karakteristik visual yang mendasari prediksi: tekstur, warna, pola, struktur permukaan, atau ciri material yang terlihat pada citra.
- Komponen tidak terdeteksi (Prediksi = "Tidak terdeteksi"): Gunakan kalimat baku yang tercantum pada instruksi user tanpa modifikasi apapun.

FORMAT OUTPUT :
Gunakan TEPAT format Toon berikut. Tidak ada teks lain di luar format ini.
Hasil[3]{Komponen,Prediksi,Status,Alasan}:
Atap,<label DTSEN>,<Status>,"<Alasan>"
Dinding,<label DTSEN>,<Status>,"<Alasan>"
Lantai,<label DTSEN>,<Status>,"<Alasan>"
"""

## SFT Dataset Generation Pipeline

### Load Metadata

In [301]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(f'Metadata tidak ditemukan: {METADATA_PATH}')

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata_all = json.load(f)

records = [
    rec for rec in metadata_all
    if rec.get('split') in {'train', 'val', 'test'}
    and rec.get('house_type') in {'single', 'multi'}
]

from collections import Counter
print('Total metadata:', len(metadata_all))
print('Total records dipakai:', len(records))
print('Split count:', Counter(rec.get('split') for rec in records))
print('House count:', Counter(rec.get('house_type') for rec in records))
print(
    "Output format:",
    "TOON"
)

records_by_split = {
    split: [rec for rec in records if rec.get('split') == split]
    for split in ['train', 'val', 'test']
}

for split, split_records in records_by_split.items():
    print(f'{split}: {len(split_records)}')

Total metadata: 14
Total records dipakai: 14
Split count: Counter({'val': 5, 'test': 5, 'train': 4})
House count: Counter({'single': 10, 'multi': 4})
Output format: TOON
train: 4
val: 5
test: 5


### Helper Functions

In [302]:
REFUSAL_KEYWORDS = [
    "maaf",
    "sorry",
    "cannot",
    "can't",
    "tidak bisa membantu",
    "tidak dapat membantu",
    "i can't",
    "unable",
    "refuse",
    "refusal",
]

In [303]:
def load_cache(path: Path) -> Dict[str, dict]:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [304]:
def save_cache(path: Path, cache: Dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

In [305]:
# ============================================================
# PARSE ALASAN JSON
# ============================================================

def parse_reason_json(
    raw: str,
):

    try:

        data = json.loads(raw)

        return {
            "atap":
                data.get(
                    "atap",
                    "",
                ),

            "dinding":
                data.get(
                    "dinding",
                    "",
                ),

            "lantai":
                data.get(
                    "lantai",
                    "",
                ),
        }

    except Exception:

        return {
            "atap": "",
            "dinding": "",
            "lantai": "",
        }

In [306]:
# ============================================================
# STATUS
# ============================================================

def resolve_status(
    prediksi,
    dtsen,
):

    pred = (
        str(prediksi)
        .strip()
        .lower()
    )

    target = (
        str(dtsen)
        .strip()
        .lower()
    )

    if pred in [
        "",
        "Tidak terdeteksi",
        "none",
    ]:

        return (
            "Tidak sesuai"
        )

    if pred == target:

        return (
            "Sesuai"
        )

    return (
        "Tidak sesuai"
    )

In [307]:
# ============================================================
# SKEMA & RESOLUSI KOMPONEN
# ============================================================

KALIMAT_SKEMA = {
    "lantai":
        "Variabel lantai tidak dapat diidentifikasi karena foto rumah tampak dalam tidak tersedia.",

    "atap":
        "Variabel atap tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia.",

    "dinding":
        "Variabel dinding tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia.",
}

KALIMAT_TIDAK_TERLIHAT = {
    "atap":
        "Variabel atap tidak dapat diidentifikasi karena komponen atap pada foto rumah tampak luar tidak terlihat.",

    "dinding":
        "Variabel dinding tidak dapat diidentifikasi karena komponen dinding pada foto rumah tampak luar tidak terlihat.",

    "lantai":
        "Variabel lantai tidak dapat diidentifikasi karena komponen lantai pada foto rumah tampak dalam tidak terlihat.",
}


def get_schema(
    images: list,
) -> str:

    view_types = {
        (
            img
            .get(
                "view_type",
                ""
            )
            .lower()
        )
        for img
        in images
    }

    if (
        "exterior"
        in view_types
        and
        "interior"
        in view_types
    ):
        return "C"

    if "exterior" in view_types:
        return "A"

    return "B"

In [308]:
def resolve_components(
    schema: str,
    prediksi: dict,
    dtsen: dict,
):

    components = {}

    for comp in [
        "atap",
        "dinding",
        "lantai",
    ]:

        pred = (
            prediksi.get(comp)
            or ""
        ).strip()

        dtsen_label = (
            dtsen.get(comp)
            or ""
        )

        # =========================
        # APAKAH BOLEH DIANALISIS
        # =========================

        can_detect = (

            (schema == "A" and comp in ["atap", "dinding"])

            or

            (schema == "B" and comp == "lantai")

            or

            (schema == "C")

        )

        # =========================
        # CASE 1
        # TIDAK BOLEH ANALISIS
        # =========================

        if not can_detect:

            components[comp] = {

                "prediksi_label":
                "Tidak terdeteksi",

                "status":
                "Tidak teridentifikasi",

                "needs_api":
                False,

                "alasan_generated":
                KALIMAT_SKEMA[comp],

            }

            continue

        # =========================
        # CASE 2
        # HARUSNYA BISA
        # TAPI VISUAL GAGAL
        # =========================

        if (
            pred == ""
            or
            pred.lower()
            == "tidak terdeteksi"
        ):

            components[comp] = {

                "prediksi_label":
                "Tidak terdeteksi",

                "status":
                "Tidak teridentifikasi",

                "needs_api":
                False,

                "alasan_generated":
                KALIMAT_TIDAK_TERLIHAT[
                    comp
                ],

            }

            continue

        # =========================
        # CASE 3
        # NORMAL
        # =========================

        status = (
            "Sesuai"
            if str(pred).lower()
            ==
            str(dtsen_label).lower()
            else
            "Tidak sesuai"
        )

        components[comp] = {

            "prediksi_label":
            pred,

            "status":
            status,

            "needs_api":
            True,

            "alasan_generated":
            "",

        }

    return components

In [309]:
# ============================================================
# BUILD TOON
# ============================================================

def build_toon(
    components,
):

    rows = [
        "Hasil[3]{Komponen,Prediksi,Status,Alasan}:"
    ]

    mapping = [
        ("atap", "Atap"),
        ("dinding", "Dinding"),
        ("lantai", "Lantai"),
    ]

    for key, label in mapping:

        c = components[key]

        rows.append(
            f'{label},'
            f'{c["prediksi_label"]},'
            f'{c["status"]},'
            f'"{c["alasan"]}"'
        )

    return "\n".join(
        rows
    )

In [310]:
def is_valid_record_images(record: dict) -> bool:
    images = record.get("images") or []
    house_type = (record.get("house_type") or "").lower().strip()

    if house_type == "multi":
        has_ext = any((img.get("view_type") or "").lower().strip() == "exterior" for img in images)
        has_int = any((img.get("view_type") or "").lower().strip() == "interior" for img in images)
        return has_ext and has_int

    return len(images) >= 1

In [311]:
def image_path_from_metadata(image_path: str) -> Path:
    return IMAGE_BASE_DIR / Path(image_path)

In [312]:
def guess_mime_type(path: Path) -> str:
    ext = path.suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext == ".webp":
        return "image/webp"
    mime, _ = mimetypes.guess_type(str(path))
    return mime or "image/jpeg"

In [313]:
def encode_image_to_data_url(path: Path) -> str:
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{guess_mime_type(path)};base64,{encoded}"

In [314]:
def sample_key(record: dict) -> str:
    payload = {
        "house_id": record.get("house_id"),
        "source_group_id": record.get("source_group_id"),
        "house_type": record.get("house_type"),
        "split": record.get("split"),
        "images": [
            (img.get("image_id"), img.get("image_path"), img.get("view_type"))
            for img in (record.get("images") or [])
        ],
    }
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

In [315]:
def select_images(record: dict) -> Tuple[Optional[dict], Optional[dict], Optional[dict]]:
    images = record.get("images") or []
    house_type = (record.get("house_type") or "").lower().strip()

    ext = None
    intr = None
    single = None

    if house_type == "multi":
        for img in images:
            vt = (img.get("view_type") or "").lower().strip()
            if vt == "exterior":
                ext = img
            elif vt == "interior":
                intr = img
    else:
        single = images[0] if images else None

    return ext, intr, single


In [316]:
def build_user_prompt_text(
    record: dict,
) -> str:

    house_id = record.get(
        "house_id",
        ""
    )

    dtsen = (
        record.get(
            "dtsen",
            {}
        )
        or {}
    )

    images = (
        record.get(
            "images"
        )
        or []
    )

    # ==========================
    # SCHEMA (pakai view_type)
    # ==========================

    schema = get_schema(
        images
    )

    if schema == "A":
        view_type = "exterior"

    elif schema == "B":
        view_type = "interior"

    else:
        view_type = "exterior + interior"

    atap = dtsen.get("atap", "-")
    dinding = dtsen.get("dinding", "-")
    lantai = dtsen.get("lantai", "-")

    header = f"""
Berikut data rumah yang harus divalidasi.

House ID : {house_id}
View Type : {view_type}

DATA REFERENSI DTSEN :

Atap : {atap}
Dinding : {dinding}
Lantai : {lantai}
""".strip()

    # ==================================================
    # SKEMA A
    # ==================================================

    if schema == "A":

        body = """
INSTRUKSI TUGAS :
 
ANALISIS KOMPONEN YANG WAJIB DIANALISIS:
Lakukan analisis material bangunan HANYA pada komponen berikut:
- Atap : Identifikasi label material atap dominan atau material dengan luas area terbesar yang menutupi bangunan. Gunakan atap segitiga (atap utama) sebagai titik acuan.Jika terdapat lembaran atap tambahan yang berada di depan dinding segitiga (seperti kanopi, teras, pelindung masuk, atau atap tambahan lain), ABAIAKAN material atap tersebut selama atap utama masih terlihat.Hanya gunakan material kanopi jika atap utama benar-benar tidak terlihat.
- Dinding : Identifikasi label material dinding dominan atau material dengan luas area terbesar yang terlihat pada tampak luar (fasad utama).

Untuk setiap komponen yang dianalisis di atas:
- Tentukan Prediksi label material resmi DTSEN.
- Tentukan Status (Sesuai / Tidak sesuai / Tidak teridentifikasi): Bandingkan hasil prediksi dengan DATA REFERENSI DTSEN dan tentukan status setiap komponen.
- Berikan penjelasan visual yang mendukung hasil prediksi untuk digunakan pada field Alasan.
 
KOMPONEN YANG TIDAK DIANALISIS:
Komponen Lantai TIDAK BOLEH DIANALISIS. Karena Variabel lantai pada DTSEN mengacu pada lantai utama di bagian dalam rumah, sedangkan input hanya berupa citra exterior.
Gunakan nilai berikut secara WAJIB untuk komponen lantai :
Prediksi : Tidak terdeteksi
Status : Tidak teridentifikasi
Alasan : "Variabel lantai tidak dapat diidentifikasi karena foto rumah tampak dalam tidak tersedia."

Output berupa format output Toon
"""

    # ==================================================
    # SKEMA B
    # ==================================================

    elif schema == "B":

        body = """
INSTRUKSI TUGAS :

ANALISIS KOMPONEN YANG WAJIB DIANALISIS:
Lakukan analisis material bangunan HANYA pada komponen lantai. Tentukan material lantai utama di dalam rumah berdasarkan material dominan (luas area terlihat terbesar). Jika lantai tertutup lapisan tambahan seperti karpet, identifikasi terlebih dahulu material dasar yang masih terlihat. Jika material dasar tidak dapat diamati, gunakan label Parket/vinil/karpet.

Untuk komponen yang dianalisis:
- Tentukan Prediksi label material resmi DTSEN pada komponen Lantai.
- Tentukan Status (Sesuai / Tidak sesuai / Tidak teridentifikasi): Bandingkan hasil prediksi dengan DATA REFERENSI DTSEN dan tentukan status komponen.
- Berikan penjelasan visual yang mendukung hasil prediksi untuk digunakan pada field Alasan.
 
KOMPONEN YANG TIDAK DIANALISIS:
Komponen Atap dan Dinding TIDAK BOLEH DIANALISIS. Karena input hanya berupa citra interior, maka material atap dan material dinding luar bangunan tidak dapat ditentukan berdasarkan definisi variabel DTSEN.
Gunakan nilai berikut secara WAJIB untuk komponen Atap dan Dinding :
Prediksi : Tidak terdeteksi
Status : Tidak teridentifikasi
Alasan :
Atap → "Variabel atap tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia."
Dinding → "Variabel dinding tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia."

Output berupa format output Toon
"""

    # ==================================================
    # SKEMA C
    # ==================================================

    else:

        body = """
INSTRUKSI TUGAS :
Gunakan seluruh gambar yang diberikan untuk membantu memahami struktur rumah dan lokasi komponen.

Untuk setiap komponen, lakukan identifikasi berdasarkan komponen fisik yang benar, dengan prioritas observasi berikut:
Atap: Tentukan material terluas atap utama bangunan. Prioritaskan atap yang menaungi bangunan utama dan abaikan kanopi atau atap tambahan jika atap utama masih terlihat.
Dinding: Tentukan material terluasdinding luar bangunan (fasad utama), bukan sekat interior, furnitur, pagar, dekorasi, atau elemen non-struktural. Pilih material dominan berdasarkan luas area terlihat terbesar.
Lantai: Tentukan material terluas lantai utama di dalam rumah (bukan teras). Jika lantai tertutup karpet atau pelapis lain, cari area yang masih memperlihatkan material dasar. Jika material dasar tidak terlihat, gunakan label Parket/vinil/karpet.
 
Untuk setiap komponen (Atap,Dinding,Lantai) yang dianalisis:
- Tentukan Prediksi menggunakan label material resmi DTSEN.
- Tentukan Status (Sesuai / Tidak sesuai / Tidak teridentifikasi): Bandingkan hasil prediksi dengan DATA REFERENSI DTSEN dan tentukan status setiap komponen.
- Berikan penjelasan visual yang mendukung hasil prediksi untuk digunakan pada field Alasan.

Output wajib mengikuti FORMAT OUTPUT Toon.
"""

    return f"""
{header}

{body}
""".strip()

In [317]:
def build_user_content(
    record: dict,
    target: str = "sft",
) -> List[dict]:

    if target not in {
        "openrouter",
        "sft",
    }:
        raise ValueError(
            "target harus 'openrouter' atau 'sft'."
        )

    house_type = (
        (
            record.get(
                "house_type"
            )
            or ""
        )
        .lower()
        .strip()
    )

    ext_img, int_img, single_img = (
        select_images(
            record
        )
    )

    content = []

    def add_image(
        img_obj: dict,
        caption: str,
    ):

        img_path = (
            image_path_from_metadata(
                img_obj[
                    "image_path"
                ]
            )
        )

        if target == "openrouter":

            content.append({
                "type":
                "image_url",

                "image_url": {
                    "url":
                    encode_image_to_data_url(
                        img_path
                    )
                }
            })

        else:

            content.append({
                "type":
                "image",

                "image":
                str(
                    img_path
                )
            })

        content.append({
            "type":
            "text",

            "text":
            caption,
        })

    # =========================
    # IMAGE
    # =========================

    if house_type == "multi":

        if ext_img:
            add_image(
                ext_img,
                "Foto tampak luar."
            )

        if int_img:
            add_image(
                int_img,
                "Foto tampak dalam."
            )

    elif single_img:

        vt = (
            single_img
            .get(
                "view_type",
                "exterior"
            )
            .lower()
        )

        add_image(
            single_img,
            (
                "Foto tampak luar."
                if vt == "exterior"
                else
                "Foto tampak dalam."
            )
        )

    # =========================
    # PROMPT
    # =========================

    content.append({

        "type":
        "text",

        "text":
        (

            build_reason_user_text(

                house_id=
                record["house_id"],

                schema=
                get_schema(
                    record["images"]
                ),

                components=
                resolve_components(

                    schema=
                    get_schema(
                        record["images"]
                    ),

                    prediksi=
                    record["prediksi"],

                    dtsen=
                    record["dtsen"],

                ),

            )

            if target
            ==
            "openrouter"

            else

            build_user_prompt_text(
                record
            )

        )

    })

    return content

In [318]:
def build_messages_for_api(
    record: dict,
) -> List[dict]:

    return [

        {
            "role":
            "system",

            "content":
            get_system_prompt(
                target="openrouter"
            ),
        },

        {
            "role":
            "user",

            "content":
            build_user_content(
                record,
                target="openrouter",
            ),
        },

    ]

In [319]:
def clean_assistant_response(
    text: Optional[str],
) -> tuple[
    Optional[dict],
    str,
]:

    if not text:

        return (
            None,
            "empty_response"
        )

    cleaned = (
        text
        .strip()
    )

    if cleaned.startswith(
        "```"
    ):

        cleaned = (
            cleaned
            .replace(
                "```json",
                ""
            )
            .replace(
                "```",
                ""
            )
            .strip()
        )

    lowered = (
        cleaned
        .lower()
    )

    if any(
        x in lowered
        for x
        in REFUSAL_KEYWORDS
    ):

        return (
            None,
            "refusal"
        )

    try:

        parsed = (
            parse_reason_json(
                cleaned
            )
        )

        return (
            parsed,
            "ok"
        )

    except Exception:

        return (
            None,
            "invalid_json"
        )

In [320]:
def get_system_prompt(
    target: str,
) -> str:

    target = target.lower()

    if target == "openrouter":
        return SYSTEM_PROMPT_REASON

    if target == "sft":
        return SYSTEM_PROMPT_SFT

    raise ValueError(
        f"target tidak dikenal: {target}"
    )

In [321]:
def build_sft_sample(
    record: dict,
    assistant_text: Optional[str] = None,
) -> dict:

    messages = [

        {
            "role":
            "system",

            "content":
            get_system_prompt(
                "sft"
            )
        },

        {
            "role":
            "user",

            "content":
            build_user_content(
                record,
                target="sft",
            )
        },

    ]

    if assistant_text is not None:

        messages.append({

            "role":
            "assistant",

            "content": [

                {
                    "type":
                    "text",

                    "text":
                    assistant_text,
                }

            ],
        })

    return {

        "id": {

            "house_id":
            record.get(
                "house_id"
            ),

            "source_group_id":
            record.get(
                "source_group_id"
            ),

            "house_type":
            record.get(
                "house_type"
            ),

            "split":
            record.get(
                "split"
            ),

        },

        "messages":
        messages,

    }

In [322]:
# ════════════════════════════════════════════════════════════
# — PARSER JSON + TOON FORMATTER (FIX NESTED OUTPUT)
# ════════════════════════════════════════════════════════════

import json
import ast


def extract_alasan(value):

    """
    Ambil field alasan saja jika model
    mengembalikan:
    {
        prediksi: ...,
        status: ...,
        alasan: ...
    }
    """

    if value is None:
        return ""

    # kalau nested dict
    if isinstance(
        value,
        dict
    ):

        return str(
            value.get(
                "alasan",
                ""
            )
        ).strip()

    return str(
        value
    ).strip()


def parse_alasan(raw: str):

    if not raw:
        return None

    try:

        raw = str(
            raw
        ).strip()

        # hapus ```json
        if raw.startswith(
            "```"
        ):

            raw = (
                raw
                .replace(
                    "```json",
                    ""
                )
                .replace(
                    "```",
                    ""
                )
                .strip()
            )

        # ======================
        # parse JSON
        # ======================

        try:

            data = json.loads(
                raw
            )

        except:

            data = ast.literal_eval(
                raw
            )

        return {

            "atap":
                extract_alasan(
                    data.get(
                        "atap",
                        ""
                    )
                ),

            "dinding":
                extract_alasan(
                    data.get(
                        "dinding",
                        ""
                    )
                ),

            "lantai":
                extract_alasan(
                    data.get(
                        "lantai",
                        ""
                    )
                ),
        }

    except Exception as e:

        print(
            f"[WARN] gagal parse: {e}"
        )

        print(
            raw[:500]
        )

        return None


# ============================================================
# TOON FORMATTER
# ============================================================

def build_toon(
    components: dict
):

    lines = [
        "Hasil[3]{Komponen,Prediksi,Status,Alasan}:"
    ]

    for comp in [

        "atap",
        "dinding",
        "lantai",

    ]:

        c = components[
            comp
        ]

        alasan = (
            str(
                c.get(
                    "alasan_generated"
                )
                or c.get(
                    "kalimat_baku"
                )
                or ""
            )
            .replace(
                "\n",
                " "
            )
            .strip()
        )

        lines.append(

            f'{comp.capitalize()},'
            f'{c["prediksi_label"]},'
            f'{c["status"]},'
            f'"{alasan}"'

        )

    return "\n".join(
        lines
    )


print(
    "✓ Parser & Toon formatter siap."
)

✓ Parser & Toon formatter siap.


### OpenRouter Call

In [323]:
session = requests.Session()


def call_openrouter(
    record: dict,
) -> Optional[dict]:

    payload = {

        "model":
        MODEL_NAME,

        "messages":
        build_messages_for_api(
            record
        ),

        "temperature":
        TEMPERATURE,

        "max_tokens":
        MAX_TOKENS,

        "reasoning": {
            "effort":
            "minimal"
        },

    }

    headers = {

        "Authorization":
        f"Bearer {API_KEY}",

        "Content-Type":
        "application/json",

    }

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            response = (
                session.post(
                    OPENROUTER_URL,
                    headers=headers,
                    json=payload,
                    timeout=TIMEOUT,
                )
            )

            if response.status_code != 200:

                last_error = (
                    f"HTTP {response.status_code}"
                )

                time.sleep(
                    RETRY_SLEEP_SEC
                    *
                    attempt
                )

                continue

            data = (
                response.json()
            )

            choice = (
                data
                .get(
                    "choices",
                    [{}],
                )[0]
            )

            message = (
                choice
                .get(
                    "message",
                    {}
                )
            )

            raw = (
                message
                .get(
                    "content",
                    ""
                )
            )

            parsed, reason = (
                clean_assistant_response(
                    raw
                )
            )

            print({

                "house_id":
                record.get(
                    "house_id"
                ),

                "status":
                reason,

                "success":
                parsed
                is not None,

            })

            if parsed:

                return (
                    parsed
                )

            last_error = (
                reason
            )

        except Exception as e:

            last_error = (
                str(
                    e
                )
            )

        time.sleep(
            RETRY_SLEEP_SEC
            *
            attempt
        )

    print(
        f"[ERROR] "
        f"house_id="
        f"{record.get('house_id')} | "
        f"{last_error}"
    )

    return None

### Generation Pipeline

In [324]:
def _generate_one_record(
    record: dict,
) -> dict:

    try:

        if not is_valid_record_images(record):

            return {
                "status": "skip",
                "reason": "invalid_images",
                "record": record,
                "assistant_text": None,
            }

        # ======================
        # schema + komponen
        # ======================

        schema = get_schema(
            record.get(
                "images",
                []
            )
        )

        components = resolve_components(
            schema,
            record.get(
                "prediksi",
                {}
            ) or {},
            record.get(
                "dtsen",
                {}
            ) or {},
        )

        # ======================
        # CALL MODEL
        # ======================

        raw_output = call_openrouter(
            record
        )
        print("\n====================")
        print(
            "HOUSE:",
            record.get(
                "house_id"
            )
        )

        print(
            "RAW OPENROUTER:"
        )

        print(
            raw_output
        )

        print("====================\n")

        if raw_output is None:

            return {
                "status": "fail",
                "reason": "openrouter_failed",
                "record": record,
                "assistant_text": None,
            }

        # ======================
        # PARSE ALASAN
        # ======================

        parsed = parse_alasan(
            raw_output
        )

        if parsed is None:

            print(
                "[WARN] parse gagal:",
                record.get(
                    "house_id"
                )
            )

            parsed = {}

        # ======================
        # ISI ALASAN
        # ======================

        for comp in [
            "atap",
            "dinding",
            "lantai",
        ]:

            if (
                components[comp]
                .get(
                    "needs_api"
                )
            ):

                value = (
                    parsed
                    .get(
                        comp,
                        ""
                    )
                )

                # MODEL BALIK DICT
                if isinstance(
                    value,
                    dict
                ):

                    value = value.get(
                        "alasan",
                        ""
                    )

                value = (
                    str(
                        value
                    )
                    .strip()
                )

                components[
                    comp
                ][
                    "alasan_generated"
                ] = value

        # ======================
        # TOON FINAL
        # ======================

        assistant_text = (
            build_toon(
                components
            )
        )

        return {
            "status": "ok",
            "reason": None,
            "record": record,
            "assistant_text": assistant_text,
            "raw_reason_output": raw_output,

        }

    except Exception as e:

        print(
            "[ERROR _generate_one_record]",
            record.get(
                "house_id"
            ),
            str(
                e
            ),
        )

        return {
            "status": "fail",
            "reason": str(
                e
            ),
            "record": record,
            "assistant_text": None,
        }

In [325]:
def read_existing_ids(output_path: Path) -> set:

    if not output_path.exists():
        return set()

    done = set()

    with open(output_path, "r", encoding="utf-8") as f:

        for line in f:

            try:

                obj = json.loads(line)

                meta = obj["id"]

                done.add((
                    meta["house_id"],
                    meta["source_group_id"],
                    meta["house_type"],
                    meta["split"],
                ))

            except Exception:
                continue

    return done 

In [326]:
def append_jsonl(
    path: Path,
    samples: List[dict],
) -> None:

    if not samples:
        return

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:

        f.writelines(
            json.dumps(
                s,
                ensure_ascii=False,
            ) + "\n"
            for s in samples
        )

In [327]:
def generate_split(
    records: List[dict],
    split_name: str,
    output_path: Path,
    cache: Dict[str, dict],
    max_workers: int = MAX_WORKERS,
    use_openrouter: bool = True,
) -> List[dict]:

    done_ids = read_existing_ids(output_path)

    pending = []
    final_samples = []

    for record in records:

        record_id = (
            record.get("house_id"),
            record.get("source_group_id"),
            record.get("house_type"),
            record.get("split"),
        )

        if record_id in done_ids:
            continue

        if not is_valid_record_images(record):
            continue

        cache_key = sample_key(record)

        if (
            use_openrouter
            and cache_key in cache
            and cache[cache_key]["status"] == "ok"
        ):

            final_samples.append(
                build_sft_sample(
                    record,
                    assistant_text=cache[cache_key]["assistant_text"],
                )
            )

            continue

        pending.append(record)

    # ====================================
    # GENERATE OPENROUTER
    # ====================================

    if use_openrouter and pending:

        with ThreadPoolExecutor(
            max_workers=max_workers
        ) as executor:

            futures = {
                executor.submit(
                    _generate_one_record,
                    record,
                ): record
                for record in pending
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"{split_name}"
            ):

                result = future.result()

                if result is None:
                    continue

                record = result["record"]

                cache_key = sample_key(record)

                cache[cache_key] = {
                    "status": result["status"],
                    "reason": result["reason"],
                    "assistant_text": result["assistant_text"],
                }

                if result["status"] == "ok":

                    final_samples.append(
                        build_sft_sample(
                            record,
                            result["assistant_text"],
                        )
                    )

    # ====================================
    # NO OPENROUTER
    # ====================================

    else:

        for record in pending:

            final_samples.append(
                build_sft_sample(
                    record,
                    assistant_text=None,
                )
            )

    # ====================================
    # SORT
    # ====================================

    final_samples.sort(
        key=lambda x: (
            x["id"]["split"],
            x["id"]["house_type"],
            x["id"]["house_id"],
        )
    )

    append_jsonl(
        output_path,
        final_samples,
    )

    return final_samples

## Run Generation

In [328]:
cache = load_cache(CACHE_PATH)

print(
    "Cache loaded:",
    len(cache)
)

# ==========================
# SORT
# ==========================

train_records = sorted(
    records_by_split["train"],
    key=lambda r: (
        r.get("house_type", ""),
        r.get("house_id", ""),
    ),
)

val_records = sorted(
    records_by_split["val"],
    key=lambda r: (
        r.get("house_type", ""),
        r.get("house_id", ""),
    ),
)

test_records = sorted(
    records_by_split["test"],
    key=lambda r: (
        r.get("house_type", ""),
        r.get("house_id", ""),
    ),
)

# ==========================
# GENERATE
# ==========================

train_samples = generate_split(
    train_records,
    "train",
    TRAIN_JSONL,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

save_cache(
    CACHE_PATH,
    cache,
)

val_samples = generate_split(
    val_records,
    "val",
    VAL_JSONL,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

save_cache(
    CACHE_PATH,
    cache,
)

test_samples = generate_split(
    test_records,
    "test",
    TEST_JSONL,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

save_cache(
    CACHE_PATH,
    cache,
)

# ==========================
# BUILD ALL
# ==========================

all_samples = []

for path in (
    TRAIN_JSONL,
    VAL_JSONL,
    TEST_JSONL,
):

    if not path.exists():
        continue

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        all_samples.extend(
            json.loads(line)
            for line in f
            if line.strip()
        )

with open(
    ALL_JSONL,
    "w",
    encoding="utf-8",
) as f:

    f.writelines(
        json.dumps(
            sample,
            ensure_ascii=False,
        )
        + "\n"
        for sample in all_samples
    )

# ==========================
# SUMMARY
# ==========================

print()

print(
    "Selesai."
)

print(
    "Train :",
    len(train_samples)
)

print(
    "Val   :",
    len(val_samples)
)

print(
    "Test  :",
    len(test_samples)
)

print(
    "All   :",
    len(all_samples)
)

print(
    "Cache :",
    len(cache)
)

print(
    "Output:",
    OUTPUT_BASE_DIR
)

Cache loaded: 0


train:  25%|██▌       | 1/4 [00:02<00:08,  2.99s/it]

{'house_id': 'H00185', 'status': 'ok', 'success': True}

HOUSE: H00185
RAW OPENROUTER:
{'atap': 'Atap rumah tampak menggunakan lembaran seng memanjang dengan pola bergelombang dan permukaan logam terlihat pada bidang atap utama.', 'dinding': 'Dinding luar rumah tampak tersusun dari papan kayu bertekstur serat kayu dan sambungan antar papan yang terlihat pada fasad.', 'lantai': ''}



train:  50%|█████     | 2/4 [00:03<00:03,  1.53s/it]

{'house_id': 'H00039', 'status': 'ok', 'success': True}

HOUSE: H00039
RAW OPENROUTER:
{'atap': 'Atap rumah terlihat menggunakan lembaran logam memanjang yang bergelombang dan permukaannya sedikit reflektif.', 'dinding': 'Dinding luar rumah tampak memiliki permukaan bidang solid dan rata yang menunjukkan konstruksi pasangan bata atau batako permanen.', 'lantai': 'Lantai dalam rumah terlihat menggunakan ubin keramik mengilap dengan pola kotak teratur dan garis nat yang jelas.'}



...

{'house_id': 'H00586', 'status': 'ok', 'success': True}

HOUSE: H00586
RAW OPENROUTER:
{'atap': '', 'dinding': '', 'lantai': 'Lantai dalam rumah terlihat menggunakan parket kayu dengan papan panjang sambungan sejajar dan tekstur serat kayu yang terlihat'}



test:  80%|████████  | 4/5 [00:05<00:01,  1.32s/it]

{'house_id': 'H00088', 'status': 'ok', 'success': True}

HOUSE: H00088
RAW OPENROUTER:
{'atap': 'Atap rumah terlihat menggunakan susunan genteng tanah liat berwarna merah kecoklatan dengan pola tumpuk gelombang.', 'dinding': 'Dinding luar rumah tampak memiliki permukaan bidang solid dan rata yang menunjukkan konstruksi pasangan bata atau batako terplester.', 'lantai': 'Lantai dalam rumah terlihat berupa permukaan tanah padat berwarna coklat tanpa lapisan akhir dan tekstur kasar alami.'}



test: 100%|██████████| 5/5 [00:06<00:00,  1.28s/it]

{'house_id': 'H00756', 'status': 'ok', 'success': True}

HOUSE: H00756
RAW OPENROUTER:
{'atap': 'Atap rumah tampak menggunakan susunan genteng dengan pola berbentuk sisik dan warna merah kecoklatan.', 'dinding': 'Dinding luar rumah tampak tersusun dari papan kayu vertikal yang memiliki tekstur serat dan sambungan antar papan.', 'lantai': ''}


Selesai.
Train : 4
Val   : 5
Test  : 5
All   : 14
Cache : 14
Output: data\sft_dataset
